
# arXiv PDF ➜ Text (OCR) — Jupyter Notebook

Batch‑download arXiv PDFs and extract text with **Tesseract OCR** while preserving a simple per‑page structure.

> This notebook was generated from your Python script and organized into runnable cells with a safer, OS‑aware configuration.



## Quick Start

1. **Set up dependencies** (one-time):
   - Install Poppler and Tesseract on your system.
   - Install Python libraries (see the next cell).  
2. **Configure paths** (Windows users usually need to set `TESSERACT_PATH` and `POPPLER_PATH`).
3. **Provide `arxiv_clean.json`** (from your Task 1) in the same folder as this notebook.
4. **Run** the "Run batch OCR" cell.


In [ ]:

# If you don't have packages, uncomment the %pip lines below and run.
# Internet might be required to install these.
# %pip install pdf2image pillow pytesseract requests

import sys, platform, os, json, time, re
from pathlib import Path
from datetime import datetime

import requests
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
print("✓ Imports loaded. Python", sys.version.split()[0], "| Platform:", platform.system())


In [ ]:

# === Configuration ===
# You can set these to point to your local installs, especially on Windows.
# If you already have Tesseract/Poppler in PATH, you can leave them as None.
TESSERACT_PATH = None  # Example: r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH   = None  # Example: r"C:\Program Files\poppler-25.07.0\Library\bin"

# Input/Output
JSON_FILE  = "arxiv_clean.json"  # file from Task 1 (list of papers)
OUTPUT_DIR = "pdf_ocr"           # output folder for PDFs and TXT

# Processing parameters
MAX_PAPERS = 10      # set to None for all
MAX_PAGES  = 5       # set to None for all pages

# --- Auto-config for Windows if you didn't set the paths ---
import platform
if platform.system() == "Windows":
    if TESSERACT_PATH is None:
        default_tesseract = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
        if os.path.exists(default_tesseract):
            TESSERACT_PATH = default_tesseract
    if POPPLER_PATH is None:
        # Try a common Poppler path; change it if your install differs
        maybe_poppler = r"C:\Program Files\poppler-25.07.0\Library\bin"
        if os.path.exists(maybe_poppler):
            POPPLER_PATH = maybe_poppler

# Apply Tesseract path if available
if TESSERACT_PATH and os.path.exists(TESSERACT_PATH):
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH
    print(f"✓ Tesseract configured at: {TESSERACT_PATH}")
else:
    print("ℹ️ Tesseract path not set (assuming it's in PATH). If OCR fails, set TESSERACT_PATH.")

# Print Poppler info
if POPPLER_PATH and os.path.exists(POPPLER_PATH):
    print(f"✓ Poppler detected at: {POPPLER_PATH}")
else:
    print("ℹ️ Poppler path not set (assuming it's in PATH). If PDF→image fails, set POPPLER_PATH.")

# Make output folders
OUTPUT_DIR = Path(OUTPUT_DIR)
PDF_DIR = OUTPUT_DIR / "pdfs"
TXT_DIR = OUTPUT_DIR / "txt_files"
for d in [OUTPUT_DIR, PDF_DIR, TXT_DIR]:
    d.mkdir(exist_ok=True)
print(f"✓ Output folders ready at: {OUTPUT_DIR.resolve()}")


In [ ]:
import json
import time
import requests
import pytesseract
from pathlib import Path
from datetime import datetime
from pdf2image import convert_from_path

# Set this to your poppler path if needed, or leave as None
POPPLER_PATH = None  # e.g., r"C:\Program Files\poppler\bin" on Windows

class ArxivPDFOCR:
    def __init__(self, json_file="arxiv_clean.json", output_dir="pdf_ocr"):
        self.json_file = json_file
        self.output_dir = Path(output_dir)
        self.pdf_dir = self.output_dir / "pdfs"
        self.txt_dir = self.output_dir / "txt_files"
        self.output_dir.mkdir(exist_ok=True)
        self.pdf_dir.mkdir(exist_ok=True)
        self.txt_dir.mkdir(exist_ok=True)

        self.papers = []
        self.processed_count = 0
        self.failed_papers = []

    def load_papers(self):
        try:
            with open(self.json_file, 'r', encoding='utf-8') as f:
                self.papers = json.load(f)
            print(f"✓ Loaded {len(self.papers)} papers from {self.json_file}")
            return True
        except FileNotFoundError:
            print(f"✗ Error: {self.json_file} not found!")
            return False
        except Exception as e:
            print(f"✗ Error loading JSON: {e}")
            return False

    @staticmethod
    def get_pdf_url(paper_id):
        paper_id = paper_id.split('v')[0]  # strip version
        return f"https://arxiv.org/pdf/{paper_id}.pdf"

    def download_pdf(self, paper_id, max_retries=3):
        pdf_url = self.get_pdf_url(paper_id)
        pdf_path = self.pdf_dir / f"{paper_id}.pdf"
        if pdf_path.exists():
            print(f"  → PDF already exists: {pdf_path.name}")
            return pdf_path

        for attempt in range(1, max_retries + 1):
            try:
                print(f"  → Downloading PDF: {pdf_url}")
                resp = requests.get(pdf_url, timeout=60, stream=True)
                resp.raise_for_status()
                with open(pdf_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"  ✓ Downloaded: {pdf_path.name}")
                return pdf_path
            except Exception as e:
                print(f"  ✗ Attempt {attempt}/{max_retries} failed: {e}")
                if attempt < max_retries:
                    time.sleep(2)
        return None

    def pdf_to_text_ocr(self, pdf_path, paper_id, max_pages=None):
        try:
            print("  → Converting PDF to images...")
            convert_kwargs = dict(dpi=200, fmt='png')
            # Only pass poppler_path if provided
            if POPPLER_PATH:
                convert_kwargs['poppler_path'] = POPPLER_PATH

            if max_pages:
                images = convert_from_path(
                    pdf_path,
                    first_page=1,
                    last_page=max_pages,
                    **convert_kwargs
                )
            else:
                images = convert_from_path(pdf_path, **convert_kwargs)

            print(f"  ✓ Converted {len(images)} pages to images")
            print(f"  → Performing OCR on {len(images)} pages...")
            full_text = []

            for i, image in enumerate(images, 1):
                print(f"    → OCR page {i}/{len(images)}...", end=' ')
                try:
                    text = pytesseract.image_to_string(image, config='--psm 6')
                    if text.strip():
                        full_text.append("\n" + "="*60)
                        full_text.append(f"PAGE {i}")
                        full_text.append("="*60 + "\n")
                        full_text.append(text)
                        print("✓")
                    else:
                        print("(empty)")
                except Exception as e:
                    print(f"✗ Error: {e}")
                    continue
            return "\n".join(full_text)
        except Exception as e:
            print(f"  ✗ Error converting PDF: {e}")
            return ""

    def process_paper(self, paper, max_pages=5):
        paper_id = paper.get('paper_id', '')
        title = paper.get('title', 'Unknown')
        print("\n" + "="*60)
        print(f"Processing: {title[:60]}...")
        print(f"Paper ID: {paper_id}")
        print("="*60)

        try:
            pdf_path = self.download_pdf(paper_id)
            if not pdf_path:
                self.failed_papers.append({'paper_id': paper_id, 'title': title, 'error': 'Failed to download PDF'})
                return False

            ocr_text = self.pdf_to_text_ocr(pdf_path, paper_id, max_pages)
            if not ocr_text.strip():
                print("  ⚠ Warning: No text extracted")

            txt_path = self.txt_dir / f"{paper_id}.txt"
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(f"arXiv Paper: {paper_id}\n")
                f.write(f"Title: {title}\n")
                f.write(f"Authors: {', '.join(paper.get('authors', []))}\n")
                f.write(f"Date: {paper.get('date', 'N/A')}\n")
                f.write(f"URL: {paper.get('url', '')}\n")
                f.write(f"OCR Date: {datetime.now().isoformat()}\n\n")
                f.write("="*60 + "\nOCR EXTRACTED TEXT\n" + "="*60 + "\n\n")
                f.write(ocr_text)

            print(f"  ✓ Saved OCR text to: {txt_path.name}")
            self.processed_count += 1
            return True
        except Exception as e:
            print(f"  ✗ Error processing paper: {e}")
            self.failed_papers.append({'paper_id': paper_id, 'title': title, 'error': str(e)})
            return False

    def process_all(self, max_papers=None, max_pages=5):
        if not self.load_papers():
            return
        papers_to_process = self.papers[:max_papers] if max_papers else self.papers
        total = len(papers_to_process)
        print("\n" + "="*60)
        print("Starting batch OCR processing")
        print(f"Total papers to process: {total}")
        print(f"Max pages per paper: {max_pages if max_pages else 'ALL'}")
        print(f"Output directory: {self.output_dir}")
        print("="*60 + "\n")

        start_time = time.time()
        for i, paper in enumerate(papers_to_process, 1):
            print(f"\n[{i}/{total}] ", end='')
            self.process_paper(paper, max_pages)
            if i < total:
                time.sleep(3)  # be kind to arXiv

        elapsed = time.time() - start_time
        print("\n" + "="*60)
        print("PROCESSING COMPLETE")
        print("="*60)
        print(f"Successfully processed: {self.processed_count}/{total} papers")
        print(f"Failed: {len(self.failed_papers)} papers")
        print(f"Total time: {elapsed/60:.1f} minutes")
        print(f"Output location: {self.txt_dir}")
        print("="*60)

        if self.failed_papers:
            log_path = self.output_dir / "failed_papers.json"
            with open(log_path, 'w', encoding='utf-8') as f:
                json.dump(self.failed_papers, f, indent=2)
            print(f"\nFailed papers logged to: {log_path}")

    def create_summary(self):
        summary_path = self.output_dir / "ocr_summary.txt"
        txt_files = list(self.txt_dir.glob("*.txt"))
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("arXiv PDF OCR Processing Summary\n")
            f.write("="*60 + "\n\n")
            f.write(f"Total OCR text files: {len(txt_files)}\n")
            f.write(f"Generated: {datetime.now().isoformat()}\n\n")
            f.write("="*60 + "\nProcessed Papers:\n" + "="*60 + "\n\n")
            for txt_file in sorted(txt_files):
                f.write(f"• {txt_file.stem}.txt\n")
        print(f"\n✓ Summary saved to: {summary_path}")


# Usage example:
if __name__ == "__main__":
    ocr = ArxivPDFOCR()
    # Process first 5 papers, max 5 pages each
    ocr.process_all(max_papers=5, max_pages=5)
    ocr.create_summary()

In [ ]:

# === Run batch OCR ===
processor = ArxivPDFOCR(json_file=JSON_FILE, output_dir=str(OUTPUT_DIR))
processor.process_all(max_papers=MAX_PAPERS, max_pages=MAX_PAGES)
processor.create_summary()
print("✓ Done")



## Troubleshooting

- **Tesseract not found**: Install Tesseract and set `TESSERACT_PATH` to the full path (Windows), or ensure it's on your system PATH (macOS/Linux).
- **Poppler not found**: Install Poppler and set `POPPLER_PATH` (Windows) or ensure `pdftoppm` is in PATH.
- **No internet** when installing with `%pip`: install packages in your environment first (outside the notebook), then run this notebook.
- **`arxiv_clean.json` missing**: Place it next to this notebook; it should contain a list of papers like:
  ```json
  [
    {"paper_id": "2501.01234v1", "title": "...", "authors": ["..."], "date": "YYYY-MM-DD", "url": "https://arxiv.org/abs/2501.01234"}
  ]
  ```
